# Dependencies

Install the required dependencies and libraries

In [ ]:
%pip install langchain langchain_core langchain-huggingface ipywidgets python-dotenv gqlalchemy langchain-memgraph langgraph

# Environment Variables and Constants

Initialize the constants and environment variables first

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="./.env")

HFH_TOKEN = os.getenv("HUGGINGFACEHUB_API_TOKEN")

SYSTEM_MSG = """
You are an AI security analyst.
You will be provided vulnerability scanner data in SARIF format with vulnerabilites detected in assets and services.
Alongside that, you will also be provided with the artifacts information such as assets information and service and architecture information and description.
Your job is to analyze the actual risk posed by the vulnerabilites by taking in the contextual information from the artifacts.
"""

HUMAN_MSG = """
Scanner: {scanner}

Assets: {assets}

Exploit Data: {exploit}

SBOM: {sbom}

Architecture: {architecture}

Service Documentation: {service_docs}

Service Catalogue: {service_catalogue}
"""

# Requirements

Install the requirements and import relevant modules

In [ ]:
import json
from pathlib import Path
from typing import Dict, List

from gqlalchemy import Memgraph
from langchain.agents import create_agent
from langchain.messages import HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_memgraph import MemgraphToolkit
from langchain_memgraph.graphs.memgraph import MemgraphLangChain
from langchain.tools import BaseTool

# Data Loaders

Methods used for loading the data

In [ ]:
# Connect to memgraph
url = os.getenv("MEMGRAPH_URL")
port = int(os.getenv("MEMGRAPH_PORT"))
memgraph = Memgraph(url, port)


In [ ]:

# clear existing data
memgraph.drop_database()

In [ ]:
def ingest_data_to_graph():
    """
    Ingests data into memgraph
    Loads the vulnerability dataset from json files into
    the graph database by running cypher queries. The data files need to be accessible
    by the memgraph or MAGE service.
    """

    # Connect to memgraph
    url = os.getenv("MEMGRAPH_URL")
    port = int(os.getenv("MEMGRAPH_PORT"))
    memgraph = Memgraph(url, port)

    # clear existing data
    memgraph.drop_database()

    # read query file content
    with open("query.cql", 'r') as f:
        query = f.read()
    

    # execute the query to ingest data
    memgraph.execute(query)


In [ ]:
ingest_data_to_graph()

# Workflow Methods

These methods setup the basic model workflow

In [ ]:
def create_prompt(sys_msg: str, hum_msg: str) -> ChatPromptTemplate:
    """
    Sets up the chat prompt to be used with the model

    Args:
        * sysm_msg (str): The system message template
        * hum_msg (str): The human message prompt

    Returns:
        `ChatPromptTemplate` object
    """

    chat_prompt = ChatPromptTemplate.from_messages([
        ("system", sys_msg),
        ("human", hum_msg)
    ])
    return chat_prompt


def load_model_from_hf(repo_id: str) -> ChatHuggingFace:
    """
    Connects to the model using the HuggingFace Inference API.

    Returns:
        A `ChatHuggingFace` model
    """

    llm = HuggingFaceEndpoint(
        repo_id=repo_id,
        huggingfacehub_api_token=os.getenv("HUGGINGFACEHUB_API_TOKEN")
    )
    model = ChatHuggingFace(llm=llm)
    return model

def create_chain(prompt: ChatPromptTemplate, model: ChatHuggingFace):
    """
    Create the chain

    Args:
        * prompt (ChatPromptTempalte): the chat prompt
        * model (ChatHuggingFace): the chat model
    
    Returns:

    """

    chain = prompt | model

    return chain

def connect_to_memgraph() -> MemgraphLangChain:
    """
    Connect to memgraph database instance

    Returns:
        `MemgraphLangChain` instance
    """

    db = MemgraphLangChain(
        url=f"{os.getenv("MEMGRAPH_DRIVER")}://{os.getenv("MEMGRAPH_URL")}:{os.getenv("MEMGRAPH_PORT")}",
        username='',
        password=''
    )
    return db

def get_memgraph_tools(db: MemgraphLangChain, model: ChatHuggingFace) -> List:
    """
    Retrieves memgraph langchain tools from memgraph toolkit

    Parameters:
        db (MemgraphLangChain): Database instance object
        model (ChatHuggingFace): LLM model object

    Returns:
        A list of memgraph tools
    """

    toolkit = MemgraphToolkit(
        db=db,
        llm=model
    )
    tools = toolkit.get_tools()
    return tools

def create_memgraph_agent():
    """
    Creates a lancghain agent provided with Memgraph toolkit.

    Returns:
        A compiled `StateGraph` that can be used for chat interactions.
    """

    model = load_model_from_hf("deepseek-ai/DeepSeek-V3.2")
    db = connect_to_memgraph()
    tools = get_memgraph_tools(db, model)

    agent_executor = create_agent(model, tools)
    
    return agent_executor

# Workflow Execution

Loads the configurations and starts the pipeline execution

In [ ]:
agent_executor = create_memgraph_agent()

In [ ]:
model = load_model_from_hf("deepseek-ai/DeepSeek-V3.2")
toolkit = MemgraphToolkit(db=db, llm=model)
tools = toolkit.get_tools()

In [ ]:
agent = create_agent(
    model=model,
    tools=tools,
    system_prompt="You are a security analyst.Use cypher queries to answer questions about the knowledge graph."
)

In [ ]:
message = {"message": [
    {"role": "user", "content": "What are the total number of vulnerability nodes in the graph."}
]}

In [ ]:
resp = agent.invoke(message)

In [ ]:
for r in resp["messages"]:
    print(r.content)

In [ ]:
# Setup the models

model = load_model_from_hf("deepseek-ai/DeepSeek-V3.2")

# Setup the prompts

prompt = create_prompt("you are a comedian", "Tell me a joke about {joke}")

# Chain the componenents

chain = create_chain(prompt, model)

# Alternatively, create agent

agent = create_agent(
    model=model,
    tools=[add],
    system_prompt="you are a helpfull assistant"
)



In [ ]:
resp = agent.invoke(
    {"messages": [{"role": "user", "content": "If I have 10 apples and jason gives me 3 more, how many do I have?"}]}
)

In [ ]:
resp